# Offline Evaluation Metrics untuk LLM

Notebook ini menggabungkan tiga notebook hands-on sesi "Online vs Offline
Metrics for LLM Applications" (bagian offline) menjadi satu, dengan **satu
dataset yang dipakai konsisten** di semua metrik teks (Exact Match, F1
token-level, fuzzy similarity, BLEU, ROUGE, semantic similarity). Sebelumnya
tiap metrik punya contoh kalimatnya sendiri-sendiri, jadi tidak bisa
dibandingkan secara adil satu sama lain.

Perbaikan lain dari versi sebelumnya:
- Normalisasi teks (strip + lowercase) diterapkan konsisten di semua metrik,
  bukan cuma sebagian.
- F1 token-level dipindah dari `set()` ke `Counter()` supaya token yang
  berulang dihitung benar (lihat Bagian 4).
- ROUGE tidak lagi pakai stemmer bahasa Inggris untuk teks Indonesia.
- Perplexity model asli pakai GPT-2 versi Indonesia, bukan GPT-2 default
  (Inggris), supaya perbandingan antar kalimat adil.
- Sentence embedding (dari notebook `Sentence_Embedding.ipynb`) digabung ke
  sini sebagai pelengkap metrik leksikal, karena keduanya menjawab pertanyaan
  yang sama: "apakah jawaban model ini bagus?" — cuma dari sudut berbeda.
- Pencarian semantik dibuat konsisten pakai cosine similarity di kedua cara
  (manual dan lewat FAISS), sebelumnya salah satu diam-diam pakai jarak
  Euclidean (L2).

Beberapa sel di bawah (perplexity model asli, sentence embedding) butuh
koneksi ke Hugging Face Hub, jadi harus dijalankan di Google Colab — di
sandbox lokal biasanya domain ini diblokir jaringannya.

In [1]:
!pip install -q rapidfuzz scikit-learn nltk rouge-score sentence-transformers faiss-cpu pandas

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 102.2 MB/s eta 0:00:00


In [2]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## 1. Klasifikasi — Accuracy & F1

Untuk tugas dengan label diskrit (klasifikasi, deteksi sentimen, dsb),
Accuracy dan F1 langsung bisa dipakai karena tidak ada ambiguitas "jawaban
mirip tapi beda kata" seperti pada teks generatif.

In [3]:
from sklearn.metrics import accuracy_score, f1_score

# Contoh: klasifikasi biner (yes/no)
y_true_cls = ["yes", "no", "yes", "no"]
y_pred_cls = ["yes", "yes", "yes", "no"]

accuracy = accuracy_score(y_true_cls, y_pred_cls)
f1 = f1_score(y_true_cls, y_pred_cls, pos_label="yes")

print(f"Accuracy: {accuracy}")
print(f"F1 score (kelas 'yes'): {f1}")

Accuracy: 0.75
F1 score (kelas 'yes'): 0.8


## 2. Dataset QA — dipakai konsisten di semua metrik di bawah

Dataset ini sengaja berisi 4 kasus dengan karakteristik berbeda, supaya
perbandingan antar metrik di ringkasan (Bagian 9) bermakna:

1. **Beda kapital saja** — harusnya dianggap benar oleh metrik manapun.
2. **Kata hilang + tambahan** — sebagian benar, sebagian meleset.
3. **Parafrase, tapi jawabannya benar** — beda kata, sama makna.
4. **Salah fakta (halusinasi)** — kata-katanya mirip gold, tapi jawabannya
   salah.

In [4]:
gold_answers = [
    "Jakarta adalah ibu kota Indonesia",
    "PSO adalah algoritma optimasi berbasis swarm",
    "Ibu kota Indonesia adalah Jakarta",
    "Ibu kota Indonesia adalah Jakarta",
]
pred_answers = [
    "Jakarta adalah ibu kota indonesia",           # Kasus 1: beda kapital saja
    "PSO adalah algoritma berbasis swarm halo",    # Kasus 2: kata hilang + tambahan
    "Jakarta merupakan ibu kota negara Indonesia", # Kasus 3: parafrase, jawaban benar
    "Ibu kota Indonesia adalah Surabaya",          # Kasus 4: salah fakta (halusinasi)
]
kasus_label = [
    "1. Beda kapital saja",
    "2. Kata hilang + tambahan",
    "3. Parafrase (jawaban benar)",
    "4. Salah fakta (halusinasi)",
]

## 3. Exact Match (EM)

Normalisasi (`strip().lower()`) diterapkan di sini dan dipakai konsisten di
semua metrik teks di bawah, supaya perbedaan angka antar metrik nanti murni
karena cara kerja metriknya sendiri — bukan karena normalisasi yang
berbeda-beda antar sel seperti di notebook sebelumnya.

In [5]:
def exact_match(pred: str, gold: str) -> int:
    '''Return 1 jika prediksi == ground truth (dinormalisasi strip+lower), 0 jika tidak.'''
    return int(pred.strip().lower() == gold.strip().lower())

matches = [exact_match(p, g) for p, g in zip(pred_answers, gold_answers)]
for label, p, g, m in zip(kasus_label, pred_answers, gold_answers, matches):
    print(f"[{label}] EM={m}")
    print(f"  Pred: {p}")
    print(f"  Gold: {g}\n")

print("EM rata-rata:", sum(matches) / len(matches))

[1. Beda kapital saja] EM=1
  Pred: Jakarta adalah ibu kota indonesia
  Gold: Jakarta adalah ibu kota Indonesia

[2. Kata hilang + tambahan] EM=0
  Pred: PSO adalah algoritma berbasis swarm halo
  Gold: PSO adalah algoritma optimasi berbasis swarm

[3. Parafrase (jawaban benar)] EM=0
  Pred: Jakarta merupakan ibu kota negara Indonesia
  Gold: Ibu kota Indonesia adalah Jakarta

[4. Salah fakta (halusinasi)] EM=0
  Pred: Ibu kota Indonesia adalah Surabaya
  Gold: Ibu kota Indonesia adalah Jakarta

EM rata-rata: 0.25


## 4. F1 Token-Level

Implementasi awal (di kedua notebook sebelumnya) memakai `set()` untuk
menghitung token yang beririsan. Masalahnya, `set()` membuang informasi
token yang berulang, jadi kalimat dengan pengulangan kata bisa dapat skor F1
sempurna padahal jelas beda panjang dan isi.

Perbaikannya memakai `Counter` (multiset), pola yang sama dengan
implementasi F1 token-level standar SQuAD.

In [6]:
from collections import Counter

def f1_token_level(pred: str, gold: str) -> float:
    pred_tokens = pred.lower().split()
    gold_tokens = gold.lower().split()

    common = sum((Counter(pred_tokens) & Counter(gold_tokens)).values())
    if common == 0:
        return 0.0

    precision = common / len(pred_tokens)
    recall = common / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

for label, p, g in zip(kasus_label, pred_answers, gold_answers):
    print(f"[{label}] F1 token-level: {f1_token_level(p, g):.3f}")

[1. Beda kapital saja] F1 token-level: 1.000
[2. Kata hilang + tambahan] F1 token-level: 0.833
[3. Parafrase (jawaban benar)] F1 token-level: 0.727
[4. Salah fakta (halusinasi)] F1 token-level: 0.800


In [7]:
# Kenapa harus Counter, bukan set() — contoh dengan token berulang
pred_dup = "kucing kucing makan"
gold_dup = "kucing makan"

def f1_token_level_versi_set(pred: str, gold: str) -> float:
    pred_tokens = set(pred.lower().split())
    gold_tokens = set(gold.lower().split())
    common = pred_tokens & gold_tokens
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

print("Pred:", pred_dup, "| Gold:", gold_dup)
print("Versi set()     (salah) :", f1_token_level_versi_set(pred_dup, gold_dup))
print("Versi Counter() (benar) :", f1_token_level(pred_dup, gold_dup))

Pred: kucing kucing makan | Gold: kucing makan
Versi set()     (salah) : 1.0
Versi Counter() (benar) : 0.8


## 5. Fuzzy String Similarity (pelengkap)

`rapidfuzz` mengukur kemiripan berbasis edit distance karakter, jadi lebih
sensitif ke typo dan variasi kapitalisasi kecil dibanding F1 token-level,
tapi tetap tidak memahami parafrase.

In [8]:
from rapidfuzz import fuzz

for label, p, g in zip(kasus_label, pred_answers, gold_answers):
    print(f"[{label}] Fuzzy ratio: {fuzz.ratio(g, p):.1f}")

[1. Beda kapital saja] Fuzzy ratio: 97.0
[2. Kata hilang + tambahan] Fuzzy ratio: 83.3
[3. Parafrase (jawaban benar)] Fuzzy ratio: 44.7
[4. Salah fakta (halusinasi)] Fuzzy ratio: 86.6


## 6. Perplexity

Bagian pertama memakai log-prob contoh manual supaya rumusnya jelas.

Bagian kedua memakai model asli. Kita pakai **GPT-2 versi Indonesia**
(`flax-community/gpt2-small-indonesian`), bukan GPT-2 default yang dilatih
korpus Inggris — mengukur perplexity teks Indonesia dengan model berbahasa
Inggris tidak adil: perplexity-nya akan tinggi bukan karena kalimatnya aneh,
tapi karena modelnya memang tidak pernah melihat bahasa Indonesia.

> Sel di bawah butuh koneksi ke Hugging Face Hub — jalankan di Google Colab.

In [9]:
import math

log_probs = [-1.2, -0.5, -0.8, -1.0]  # contoh log-prob per token (natural log)
avg_log_prob = sum(log_probs) / len(log_probs)
perplexity = math.exp(-avg_log_prob)
print("Perplexity (contoh manual):", perplexity)

Perplexity (contoh manual): 2.398875293967098


In [10]:
# Perlu koneksi ke Hugging Face Hub -> jalankan di Google Colab
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "flax-community/gpt2-small-indonesian"  # GPT-2 versi Indonesia
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

def hitung_perplexity(text: str) -> float:
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return math.exp(outputs.loss.item())

kalimat_benar = "Jakarta adalah ibu kota Indonesia."
kalimat_typo = "Jakarta merupakan pusat pemerintaha Indonesia."  # ada typo "pemerintaha"

print("Perplexity (kalimat rapi) :", hitung_perplexity(kalimat_benar))
print("Perplexity (ada typo)     :", hitung_perplexity(kalimat_typo))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/863 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  510MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: flax-community/gpt2-small-indonesian
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Perplexity (kalimat rapi) : 17.286185156731033
Perplexity (ada typo)     : 145.15951932983745


## 7. BLEU Score

Dihitung dari dataset yang sama di Bagian 2 (bukan kalimat hardcode
terpisah seperti sebelumnya), dengan normalisasi lower+strip yang sama
seperti EM dan F1, supaya adil dibandingkan di ringkasan.

In [11]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1

for label, p, g in zip(kasus_label, pred_answers, gold_answers):
    reference = nltk.word_tokenize(g.strip().lower())
    candidate = nltk.word_tokenize(p.strip().lower())
    bleu = sentence_bleu([reference], candidate, smoothing_function=smooth)
    print(f"[{label}] BLEU: {bleu:.3f}")

[1. Beda kapital saja] BLEU: 1.000
[2. Kata hilang + tambahan] BLEU: 0.254
[3. Parafrase (jawaban benar)] BLEU: 0.103
[4. Salah fakta (halusinasi)] BLEU: 0.669


## 8. ROUGE Score

`use_stemmer=True` di `rouge-score` memakai Porter stemmer bahasa Inggris —
tidak cocok untuk teks Indonesia (imbuhan bisa terpotong salah). Di sini
dipakai `use_stemmer=False`.

In [12]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

for label, p, g in zip(kasus_label, pred_answers, gold_answers):
    scores = scorer.score(g, p)
    print(f"[{label}]")
    print(f"  ROUGE-1: {scores['rouge1']}")
    print(f"  ROUGE-L: {scores['rougeL']}\n")

[1. Beda kapital saja]
  ROUGE-1: Score(precision=1.0, recall=1.0, fmeasure=1.0)
  ROUGE-L: Score(precision=1.0, recall=1.0, fmeasure=1.0)

[2. Kata hilang + tambahan]
  ROUGE-1: Score(precision=0.8333333333333334, recall=0.8333333333333334, fmeasure=0.8333333333333334)
  ROUGE-L: Score(precision=0.8333333333333334, recall=0.8333333333333334, fmeasure=0.8333333333333334)

[3. Parafrase (jawaban benar)]
  ROUGE-1: Score(precision=0.6666666666666666, recall=0.8, fmeasure=0.7272727272727272)
  ROUGE-L: Score(precision=0.5, recall=0.6, fmeasure=0.5454545454545454)

[4. Salah fakta (halusinasi)]
  ROUGE-1: Score(precision=0.8, recall=0.8, fmeasure=0.8000000000000002)
  ROUGE-L: Score(precision=0.8, recall=0.8, fmeasure=0.8000000000000002)



## 9. Ringkasan — satu pasangan jawaban, metrik berbeda-beda

Jalankan semua sel di atas dulu, lalu lihat tabel gabungannya di sini.

Yang menarik untuk diperhatikan: **Kasus 3** (parafrase yang jawabannya
benar) mendapat skor rendah di hampir semua metrik leksikal, sementara
**Kasus 4** (salah fakta) justru mendapat skor tinggi karena kata-katanya
mirip gold. Ini bukan sekadar bug di satu metrik — semua metrik di atas
memang cuma membandingkan kata, bukan makna maupun kebenaran faktual.

In [13]:
import pandas as pd

rows = []
for label, p, g in zip(kasus_label, pred_answers, gold_answers):
    reference = nltk.word_tokenize(g.strip().lower())
    candidate = nltk.word_tokenize(p.strip().lower())
    rows.append({
        "Kasus": label,
        "EM": exact_match(p, g),
        "F1 token": round(f1_token_level(p, g), 3),
        "Fuzzy": round(fuzz.ratio(g, p), 1),
        "BLEU": round(sentence_bleu([reference], candidate, smoothing_function=smooth), 3),
        "ROUGE-L": round(scorer.score(g, p)['rougeL'].fmeasure, 3),
    })

df_ringkasan = pd.DataFrame(rows)
df_ringkasan

,Kasus,EM,F1 token,Fuzzy,BLEU,ROUGE-L
0,1. Beda kapital saja,1,1.000,97.0,1.000,1.000
1,2. Kata hilang + tambahan,0,0.833,83.3,0.254,0.833
2,3. Parafrase (jawaban benar),0,0.727,44.7,0.103,0.545
3,4. Salah fakta (halusinasi),0,0.800,86.6,0.669,0.800


## 10. Semantic Similarity — Sentence Embedding

Semua metrik di atas membandingkan kata. Sentence embedding membandingkan
makna. Kita pakai model multilingual
(`paraphrase-multilingual-MiniLM-L12-v2`) supaya bisa langsung dites ke
pasangan Kasus 3 (parafrase) dan Kasus 4 (salah fakta) yang tadi
bermasalah.

> Sel ini butuh koneksi ke Hugging Face Hub — jalankan di Google Colab.

In [14]:
# Perlu koneksi ke Hugging Face Hub -> jalankan di Google Colab
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

for label, p, g in zip(kasus_label, pred_answers, gold_answers):
    vecs = model_emb.encode([g, p])
    sim = cosine_similarity([vecs[0]], [vecs[1]])[0][0]
    print(f"[{label}] Cosine similarity: {sim:.3f}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  510MB            

model.safetensors: downloading bytes:           |  0.00B            

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[1. Beda kapital saja] Cosine similarity: 0.998
[2. Kata hilang + tambahan] Cosine similarity: 0.869
[3. Parafrase (jawaban benar)] Cosine similarity: 0.993
[4. Salah fakta (halusinasi)] Cosine similarity: 0.682


**Cara membaca hasilnya:** bandingkan angka di atas dengan tabel BLEU/ROUGE
di Bagian 9.

- Kalau *cosine similarity* Kasus 3 jauh lebih tinggi dari BLEU/ROUGE-nya,
  itu tanda embedding berhasil menangkap makna yang terlewat oleh metrik
  leksikal.
- Kalau *cosine similarity* Kasus 4 juga ikut tinggi, itu tanda embedding
  sendirian belum cukup untuk mendeteksi salah fakta — makna kalimatnya
  memang mirip ("ibu kota Indonesia adalah X"), meski X-nya salah. Untuk
  kasus ini dibutuhkan pengecekan lain (LLM-as-a-Judge atau fact-checking
  terpisah, di luar cakupan notebook ini).

## 11. Pencarian Semantik (Semantic Search)

Metode ini yang jadi dasar retrieval di sistem RAG. Pencarian di bawah
sengaja dibuat memakai **cosine similarity secara konsisten** — baik cara
manual maupun lewat FAISS — supaya urutan hasilnya sama-sama berdasarkan
sudut antar vektor. Sebelumnya salah satu bagian diam-diam memakai jarak
Euclidean (L2), yang bisa menghasilkan urutan berbeda dari perhitungan
cosine manual di atasnya.

> Sel ini melanjutkan `model_emb` dari Bagian 10 — jalankan di Google
> Colab.

In [15]:
import numpy as np

kalimat = [
    "Harga emas hari ini mengalami kenaikan",
    "Nasabah dapat mengajukan pinjaman KCA",
    "Emas Antam naik dua persen",
]
query = model_emb.encode(["harga logam mulia hari ini"])
docs = model_emb.encode(kalimat)

skor = cosine_similarity(query, docs)[0]
urutan = np.argsort(-skor)
for i in urutan:
    print(f"{skor[i]:.3f}  {kalimat[i]}")

0.674  Harga emas hari ini mengalami kenaikan
0.450  Emas Antam naik dua persen
-0.007  Nasabah dapat mengajukan pinjaman KCA


In [16]:
import faiss

def normalize(v):
    return v / np.linalg.norm(v, axis=1, keepdims=True)

docs_norm = normalize(docs).astype("float32")
query_norm = normalize(query).astype("float32")

d = docs_norm.shape[1]
index = faiss.IndexFlatIP(d)  # inner product pada vektor ternormalisasi = cosine similarity
index.add(docs_norm)

D, I = index.search(query_norm, k=3)
print("Skor (cosine, makin besar makin mirip):", D)
print("Urutan indeks dokumen:", I)

Skor (cosine, makin besar makin mirip): [[ 0.67431337  0.45048338 -0.00736667]]
Urutan indeks dokumen: [[0 2 1]]


## 12. Online Metrics — demo struktur data (ilustrasi)

Bagian ini cuma ilustrasi struktur data untuk like/dislike/regenerate,
**bukan sistem sungguhan** — nilainya di sini akan bertambah terus tiap sel
dijalankan ulang, jadi jangan dijadikan acuan angka apa pun. Implementasi
nyata (database, endpoint API, perhitungan rate yang benar, latency) ada di
notebook terpisah: `Online_Metrics_LLM_with_MySQL.ipynb`.

In [17]:
user_feedback = {"like": 0, "dislike": 0}
regenerate_count = 0

def record_feedback(feedback_type: str):
    if feedback_type in user_feedback:
        user_feedback[feedback_type] += 1

def regenerate_response():
    global regenerate_count
    regenerate_count += 1

record_feedback("like")
record_feedback("dislike")
regenerate_response()

print("Feedback:", user_feedback)
print("Jumlah regenerasi:", regenerate_count)

Feedback: {'like': 1, 'dislike': 1}
Jumlah regenerasi: 1


## Kesimpulan

- Accuracy/F1 cocok untuk label diskrit, tapi tidak langsung berlaku untuk
  teks generatif panjang.
- EM, F1 token-level, fuzzy ratio, BLEU, dan ROUGE semuanya membandingkan
  **kata**, dengan cara masing-masing yang berbeda — makanya satu pasangan
  jawaban bisa dapat angka yang jauh berbeda di tiap metrik.
- Sentence embedding menutup sebagian celah itu dengan membandingkan
  **makna**, tapi juga bukan solusi tunggal: jawaban yang salah fakta bisa
  saja tetap "mirip secara makna" dengan jawaban yang benar.
- Kesimpulan materi tetap berlaku: idealnya beberapa metrik dipakai
  berdampingan, bukan mengandalkan satu angka saja.

---
*Notebook ini disusun sebagai bagian dari pembelajaran mandiri materi
"Online vs Offline Metrics for LLM Applications" — rubythalib.ai.*